 Molecular Dynamics (MD) Simulation (OpenMM)
OverviewWhile molecular docking provides a static snapshot of binding affinity, it lacks the dynamic physical realities of a cellular environment. This notebook elevates the pipeline by performing Molecular Dynamics (MD) simulations using the GPU-accelerated OpenMM toolkit. By immersing the system in a simulated explicit water environment at physiological temperature and pressure, this step verifies the thermodynamic stability and structural integrity of the biological target over time.
Workflow & Methodology

Structural Repair (PDBFixer): Raw crystallographic structures often lack terminal atoms or unresolved loops. The script utilizes PDBFixer to automatically model missing residues, rebuild missing heavy atoms, and protonate the system to a physiological pH of 7.4.
System Solvation & Ionization: The repaired macromolecule is enclosed in a cubic solvent box using the TIP3P explicit water model (1.0 nm padding). The system is then neutralized and brought to a physiological salt concentration (0.15 M NaCl).
Forcefield Application: Employs the highly validated AMBER14 forcefield to define the physics, bonds, and van der Waals interactions of the system.
Energy Minimization: Runs an initial geometry optimization to resolve any steric clashes or high-energy atomic overlaps introduced during the solvation phase.
NPT Ensemble Simulation: The system is simulated under Isothermal-Isobaric (NPT) conditions:
Temperature: Maintained at 300 K using a Langevin Middle Integrator.
Pressure: Maintained at 1 atm using a Monte Carlo Barostat.


Trajectory Logging:
Captures the thermodynamic state (Potential Energy, Temperature, Volume) and 3D atomic coordinates into a trajectory (.dcd) and log file (.csv) at regular intervals.

### Dependencies

Python Libraries:
openmm, openmm.app, pdbfixer, os, sys, google.colab.drive.



In [1]:
!pip install pdbfixer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.9/665.9 kB 14.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 70.5 MB/s eta 0:00:00
  Created wheel for pdbfixer: filename=pdbfixer-1.12.0-py3-none-any.whl size=681683 sha256=56ac1626ebaf4a865de4d00f364337d91e54ecf50afe0ab8d2af1fd2cfef39af
  Stored in directory: /root/.cache/pip/wheels/79/ec/63/2ad240b7fac835f490bd1ecb748da87c99ae65618770c73a98
Successfully built pdbfixer


In [2]:
import os
import openmm as mm
import openmm.app as app
import openmm.unit as unit
from sys import stdout
from pdbfixer import PDBFixer  # <-- We added PDBFixer here!
from google.colab import drive

# --- 1. Mount Google Drive ---
print("Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)

# --- 2. Define Paths ---
BASE_DIR = "/content/drive/MyDrive/Docking_Pipeline"
PREP_DIR = os.path.join(BASE_DIR, "prep_data")
MD_DIR = os.path.join(BASE_DIR, "md_results")
os.makedirs(MD_DIR, exist_ok=True)

PDB_ID = "1HSG"
protein_pdb = os.path.join(PREP_DIR, f"{PDB_ID}_clean.pdb")
output_traj = os.path.join(MD_DIR, f"{PDB_ID}_trajectory.dcd")
output_log = os.path.join(MD_DIR, f"{PDB_ID}_md_log.csv")

print("\n--- Starting Molecular Dynamics Setup ---")

if not os.path.exists(protein_pdb):
    print(f"❌ Error: Cannot find {protein_pdb}")
else:
    # --- 3. Repair the Protein with PDBFixer ---
    print("Fixing missing atoms, residues, and adding hydrogens...")
    fixer = PDBFixer(filename=protein_pdb)
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.4) # Add hydrogens for pH 7.4
    print("✅ Protein repaired successfully.")

    # --- 4. Load Forcefield and Solvate ---
    print("Loading AMBER14 forcefield...")
    forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

    print("Adding water box and neutralizing ions...")
    # Notice we now use fixer.topology and fixer.positions!
    modeller = app.Modeller(fixer.topology, fixer.positions)
    modeller.addSolvent(forcefield, padding=1.0*unit.nanometers, ionicStrength=0.15*unit.molar)

    # --- 5. Create the System ---
    print("Creating the physics system (NPT ensemble)...")
    system = forcefield.createSystem(modeller.topology, nonbondedMethod=app.PME,
                                     nonbondedCutoff=1.0*unit.nanometers, constraints=app.HBonds)

    integrator = mm.LangevinMiddleIntegrator(300*unit.kelvin, 1/unit.picosecond, 0.002*unit.picoseconds)
    system.addForce(mm.MonteCarloBarostat(1*unit.atmospheres, 300*unit.kelvin))

    simulation = app.Simulation(modeller.topology, system, integrator)
    simulation.context.setPositions(modeller.positions)

    # --- 6. Energy Minimization ---
    print("\nMinimizing energy (removing structural clashes)...")
    simulation.minimizeEnergy()
    print("✅ Minimization complete.")

    # --- 7. Run Simulation ---
    steps = 5000
    print(f"\nRunning MD simulation for {steps} steps (Grab a coffee, this will take a moment!)...")

    simulation.reporters.append(app.DCDReporter(output_traj, 1000))
    simulation.reporters.append(app.StateDataReporter(output_log, 1000, step=True,
                                                      potentialEnergy=True, temperature=True, volume=True))
    simulation.reporters.append(app.StateDataReporter(stdout, 1000, step=True,
                                                      potentialEnergy=True, temperature=True))

    simulation.step(steps)

    print("\n--- MD Simulation Complete! ---")
    print(f"Trajectory saved to: {output_traj}")
    print(f"Data log saved to: {output_log}")

Connecting to Google Drive...
Mounted at /content/drive

--- Starting Molecular Dynamics Setup ---
Fixing missing atoms, residues, and adding hydrogens...
✅ Protein repaired successfully.
Loading AMBER14 forcefield...
Adding water box and neutralizing ions...
Creating the physics system (NPT ensemble)...

Minimizing energy (removing structural clashes)...
✅ Minimization complete.

Running MD simulation for 5000 steps (Grab a coffee, this will take a moment!)...
#"Step","Potential Energy (kJ/mole)","Temperature (K)"
1000,-789117.9576086139,254.95122047762976
2000,-768954.1472374815,290.44005896146024
3000,-764256.1527227544,301.1865730105225
4000,-760500.8270933963,300.23018394384826
5000,-761824.8309189806,300.37787329466585

--- MD Simulation Complete! ---
Trajectory saved to: /content/drive/MyDrive/Docking_Pipeline/md_results/1HSG_trajectory.dcd
Data log saved to: /content/drive/MyDrive/Docking_Pipeline/md_results/1HSG_md_log.csv
